# Notebook 07 — Evidence Sufficiency, Policy Enforcement & Human Approval

## Learning Objective

Understand how enterprise AI agents can make evidence-grounded recommendations while keeping business decisions and consequential actions under deterministic controls.

**Core principle: The LLM proposes facts. Trusted systems verify facts. Deterministic policies make decisions. Authorized actors execute actions.**

## Why This Matters

An AI agent may correctly retrieve a claim and still recommend an unsupported payout. Accurate data retrieval does not automatically establish coverage, eligibility, or approval authority.

We must distinguish:

1. **Evidence sufficiency:** Do we have verified facts supporting the recommendation?
2. **Policy enforcement:** Does the recommendation comply with business rules?
3. **Action authorization:** Is the actor permitted to approve or execute the action?

## What We'll Build

Using a synthetic insurance investigation scenario, we'll:

- Evaluate claim eligibility using verified evidence.
- Apply deterministic payout thresholds.
- Require supervisor review for higher-value claims.
- Prevent AI agents from approving or executing payments.
- Test missing evidence, unsupported recommendations, and unauthorized actions.

## Key Security Concepts

- **Untrusted inputs:** Model outputs and retrieved documents are not authoritative facts.
- **Evidence provenance:** A source reference must be independently validated before its claims are trusted.
- **Deterministic guardrails:** Business rules should be enforced in code rather than left to model judgment.
- **Human-in-the-loop:** Consequential decisions may require explicit human approval.
- **Least privilege:** An investigation agent may recommend actions without possessing permission to execute them.

## Expected Outcome

By the end, we should understand why a recommendation can be plausible but unsupported, why a policy-compliant decision can still rely on untrusted inputs, and how to separate evidence verification, business-policy evaluation, and execution authority.

**Key takeaway: A well-structured model output is not necessarily a verified fact, and a valid recommendation is not authorization to act.**

In [1]:
from decimal import Decimal

from investigation_agent_security_lab.decision_policy import (
    Coverage,
    VerifiedEvidence,
    evaluate_payout_policy,
    authorize_agent_action,
)

scenarios = {
    "missing_evidence": (
        Decimal("4200"),
        VerifiedEvidence(Coverage.UNKNOWN, False, None),
    ),
    "covered_small": (
        Decimal("4200"),
        VerifiedEvidence(
            Coverage.COVERED, True, "verified-fixture-001"
        ),
    ),
    "covered_large": (
        Decimal("8000"),
        VerifiedEvidence(
            Coverage.COVERED, True, "verified-fixture-002"
        ),
    ),
    "not_covered": (
        Decimal("4200"),
        VerifiedEvidence(
            Coverage.NOT_COVERED, True, "verified-fixture-003"
        ),
    ),
}

for name, (amount, evidence) in scenarios.items():
    outcome = evaluate_payout_policy(
        amount=amount,
        evidence=evidence,
    )
    print(name, "→", outcome.value)

missing_evidence → NEEDS_MORE_EVIDENCE
covered_small → ELIGIBLE_FOR_ADJUSTER_REVIEW
covered_large → REQUIRES_SUPERVISOR_APPROVAL
not_covered → NOT_COVERED


In [2]:
for action in ["recommend", "approve", "execute_payment"]:
    outcome = authorize_agent_action(action)
    print(action, "→", outcome.value if outcome else "PERMITTED")

recommend → PERMITTED
approve → ACTION_DENIED
execute_payment → ACTION_DENIED


In [3]:
agent_recommendation = {
    "claim_id": "C-1042",
    "action": "recommend",
    "recommendation": "Approve a $4,200 payout",
}

In [5]:
from investigation_agent_security_lab.decision_policy import (
    Coverage,
    VerifiedEvidence,
    evaluate_payout_policy,
    Decision,
)

current_evidence = VerifiedEvidence(
    coverage=Coverage.UNKNOWN,
    evidence_complete=False,
    source_id=None,
)

policy_decision = evaluate_payout_policy(
    amount=Decimal("4200"),
    evidence=current_evidence,
)

assert policy_decision == Decision.NEEDS_MORE_EVIDENCE